# Part 4b - Perplexity (PPL) Sanity Check (INT8)

Compute log-perplexity for the **INT8 bitsandbytes** Llama-2-7B-chat checkpoint
(produced by `00_quantize.ipynb`) on a small neutral eval set. Compared against the
FP16 baseline run in `tml/04b_perplexity.ipynb`, the gap shows how much language-modeling
fidelity bnb 8-bit costs us, and feeds the Perplexity-Filter threshold in Part 6.

**Outputs**: `results_part4b/summary.csv`, `results_part4b/raw.json`.


In [1]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter (tml/ vs repo root).
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 100 harmful goals, 100 benign goals, 300 judge rows


In [2]:
import os, json, gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PPL_TEXTS = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models are trained on large datasets using gradient descent.",
    "The weather today is sunny with a chance of afternoon thunderstorms.",
    "Natural language processing enables computers to understand human text.",
    "The economy grew by three percent in the last quarter according to reports.",
    "Quantum computing promises to solve problems that are intractable for classical computers.",
    "The patient showed significant improvement after the new treatment protocol.",
    "Renewable energy sources accounted for forty percent of total electricity generation.",
]

@torch.no_grad()
def compute_ppl(model, tokenizer, texts, max_length=128):
    total_nll, total_tokens = 0.0, 0
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt",
                           truncation=True, max_length=max_length)
        input_ids = inputs["input_ids"].to(model.device)
        outputs = model(input_ids, labels=input_ids)
        total_nll    += outputs.loss.item() * input_ids.shape[1]
        total_tokens += input_ids.shape[1]
    return float(np.exp(total_nll / total_tokens))

print("Computing PPL for INT8 Llama-2-7B-chat (GPTQModel)...")
# Load the locally pre-quantized GPTQ checkpoint from 00_quantize.ipynb via
# gptqmodel's loader (HF transformers can also load this, but GPTQModel.load
# is the canonical path and handles kernel selection).
from gptqmodel import GPTQModel
QUANT_DIR = "./quantized_int8"
base = GPTQModel.load(QUANT_DIR, device_map="auto")
tok  = AutoTokenizer.from_pretrained(QUANT_DIR)
ppl_quant = compute_ppl(base, tok, PPL_TEXTS)
print(f"  PPL = {ppl_quant:.2f}")

del base, tok
gc.collect()
torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!


Computing PPL for FP16 Llama-2-7B-chat...


[2026-06-04 14:12:02] INFO modeling.py:987: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:05<00:05,  5.20s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.04s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.36s/it]

  PPL = 20.94


In [3]:
os.makedirs("results_part4b", exist_ok=True)

summary_df = pd.DataFrame([{
    "Model Variant":    "INT8 (GPTQModel)",
    "Perplexity (PPL)": round(ppl_quant, 2),
}])
summary_df.to_csv("results_part4b/summary.csv", index=False)

with open("results_part4b/raw.json", "w") as f:
    json.dump({
        "model":      "meta-llama/Llama-2-7b-chat-hf",
        "checkpoint": "./quantized_int8",
        "dtype":      "int8-gptq",
        "eval_set":   PPL_TEXTS,
        "ppl":        ppl_quant,
    }, f, indent=2)

print("\n### Part 4b summary ###")
print(summary_df.to_string(index=False))



### Part 4b summary ###
  Model Variant  Perplexity (PPL)
FP16 (baseline)             20.94
